# 04 Expert review and audits

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/claude/awesome-archimedes-kvh4c7/notebooks/04_expert_review_and_audits.ipynb)

Early version of notebook 04 of the organization plan (Stage C finishes it). No GPU is needed. Run `03_zero_shot.ipynb` first: this notebook reads the baseline it wrote.

**Safe to run and re-run.** It reads the shared Drive and only creates new files, never overwriting one.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, clones this branch, installs `coralid[viz]` | 1-3 min | GitHub | nothing on Drive |
| Config | loads the run, the baseline and the manifest | seconds | `<run>/predictions/...__all.parquet`, `<run>/manifest/manifest.parquet` | nothing |
| Portrait impact, expert set | every expert row on a portrait photo: label, legacy prediction, new prediction | seconds | `assaf_expert_test_set.csv` | nothing |
| Portrait impact, Millepora audit | the 129 audit rows with their orientation, new prediction and a re-review flag | seconds | `v2_pipeline/millepora_sharp_audit.csv` | `<run>/reports/portrait_impact_<date>/` (a new folder) |
| Old audit against the baseline | the old judgments against the new predictions, as a non-expert agreement range | seconds | the above | nothing |
| Blind review sheet | a random, mixed, shuffled sheet with no model output; a subset for Assaf; the hidden key | 2-5 min | photos, mask JSONs | `<run>/reviews/exports/` (sheets and PDFs to hand out) and `<run>/reviews/keys/` (the answer key, **do not share**) |
| Scoring | after the reviews come back: precision, miss rate and recall with intervals, analyst against expert | seconds | `<run>/reviews/completed/` | `<run>/reports/blind_score_<name>_<date>/` (a new folder) |

**Why the portrait rows matter.** The legacy code cut the wrong region on the 221 portrait photos (1920x2560 frames). Assaf's expert labels and the old Millepora audit were judged on crops cut by that code. For portrait photos the judgment may describe a different-looking crop than the one `coralid` cuts now, so those rows are listed and flagged for re-review, and their old judgments are never reused.

**The Millepora figure** (66 yes, 46 no, 17 unsure, so 58.9% of the decided) was a **non-expert** analyst's opinion, who saw the model's answer, on V2's 24-genus predictions and on partly mis-cut crops. It is provisional and is not the 20-genus precision.

**What to paste back to the next session:** the printed counts of the portrait-impact cells, the agreement range, and (after the reviews) the scoring.

In [ ]:
import glob
import importlib
import importlib.util
import os
import site
import subprocess
import sys
from importlib import metadata

from google.colab import drive

# ================================ SETTINGS: the only place to edit ================================
# The Drive folder "CoralsMasking" (the root of the shared data):
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"
BRANCH = "claude/awesome-archimedes-kvh4c7"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "viz"
# ==================================================================================================

drive.mount("/content/drive")
os.environ["CORALID_DATA_ROOT"] = DATA_ROOT  # coralid reads the data location from here
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Edit DATA_ROOT above. If the folder was shared with you, "
        "add a shortcut to it in My Drive first."
    )

WATCH = ["numpy", "pandas", "pyarrow", "pillow", "opencv-python", "opencv-python-headless"]
WATCH += ["torch", "torchvision", "open-clip-torch", "transformers", "matplotlib"]
before = {d.metadata["Name"].lower().replace("_", "-"): d.version for d in metadata.distributions()}

if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # already cloned: update it
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
steps.append([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"])
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)

# An editable install adds a .pth file, and a running interpreter reads those only at startup:
# without this, `import coralid` fails in the session that just installed it.
site_dirs = site.getsitepackages() if hasattr(site, "getsitepackages") else []
for site_dir in site_dirs:
    for pth in glob.glob(os.path.join(site_dir, "__editable__*coralid*.pth")):
        site.addpackage(site_dir, os.path.basename(pth), None)
importlib.invalidate_caches()
if importlib.util.find_spec("coralid") is None:  # last resort: use the clone's src folder directly
    sys.path.insert(0, f"{REPO_DIR}/src")

after = {d.metadata["Name"].lower().replace("_", "-"): d.version for d in metadata.distributions()}
replaced = {
    k: (before[k], after.get(k)) for k in WATCH if k in before and before[k] != after.get(k)
}

print("\nInstalled versions (pip metadata):")
for name in ("opencv-python", "opencv-python-headless", "torch", "open-clip-torch", "matplotlib"):
    print(f"  {name:24} {after.get(name)}")
if (
    after.get("opencv-python")
    and after.get("opencv-python-headless")
    and after["opencv-python"] != after["opencv-python-headless"]
):
    print("  WARNING: opencv-python and opencv-python-headless differ in version;")
    print("  they share one cv2 folder.")

import coralid  # noqa: E402  (importable only after the install above)

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
print(f"\ncoralid {coralid.__version__} at commit {commit} (branch {BRANCH})")

if replaced:
    raise RuntimeError(
        "pip replaced packages that this Python session may already have loaded: "
        + ", ".join(f"{k} {a} -> {b}" for k, (a, b) in replaced.items())
        + "\n\n>>> RESTART THE RUNTIME now (Runtime > Restart session), "
        "then run this cell again and continue. <<<"
    )
print("\nSetup done. No restart needed.")

In [ ]:
import json
import logging

import pandas as pd

from coralid.baseline import baseline_paths
from coralid.config import load_config
from coralid.manifest import read_manifest

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
GENUS = "Millepora"
EXPERT_CSV = cfg.run_dir / "assaf_expert_test_set.csv"
AUDIT_CSV = cfg.run_dir / "v2_pipeline" / "millepora_sharp_audit.csv"
paths = baseline_paths(cfg)
for label, path in {
    "manifest": cfg.manifest_path,
    "baseline": paths.predictions,
    "expert set": EXPERT_CSV,
    "Millepora audit": AUDIT_CSV,
}.items():
    print(f"  {label:16} {'ok     ' if path.exists() else 'MISSING'} {path}")
if not paths.predictions.exists():
    raise FileNotFoundError("The baseline is missing: run notebooks/03_zero_shot.ipynb first.")

manifest = read_manifest(cfg.manifest_path)
predictions = pd.read_parquet(paths.predictions)
print(f"{len(predictions)} baseline crops on {predictions['image_name'].nunique()} photos")

## Portrait impact on the expert set

Every row of `assaf_expert_test_set.csv` on a portrait photo, with Assaf's label and status, the prediction stored in the CSV (legacy) and the new baseline's. The summary says how many of the "unidentifiable" rows are among them, which separates "mis-cut" from "blurred" as the reason for an unidentifiable call.

In [ ]:
from coralid.evaluation import load_expert_set, load_legacy_predictions
from coralid.portrait_impact import expert_portrait_impact

expert = load_expert_set(EXPERT_CSV, cfg.zero_shot.label_set)
legacy_predictions = load_legacy_predictions(EXPERT_CSV)
expert_rows, expert_summary = expert_portrait_impact(
    expert, legacy_predictions, predictions, manifest
)

s = expert_summary
print(f"expert rows: {s['n_expert_rows']}, by orientation {s['rows_by_orientation']}")
print(
    f"on portrait photos: {s['portrait_rows']} rows = {s['portrait_share_of_expert_rows']:.1%} "
    f"of the expert rows (portrait photos are {s['portrait_share_of_photos']:.1%} of all photos)"
)
print(
    f"unidentifiable: {s['unidentifiable_on_portrait']} of {s['unidentifiable_total']} "
    "unidentifiable rows are on portrait photos"
)
print("status by orientation:", json.dumps(s["status_by_orientation"]))
print(
    "portrait rows: in the new baseline",
    s["portrait_in_new_baseline"],
    "| prediction changed",
    s["portrait_prediction_changed"],
)
print(
    "portrait rows scored (named genus): legacy", s["portrait_legacy"], "| new", s["portrait_new"]
)
print(
    "landscape rows whose prediction changed:",
    s["landscape_prediction_changed"],
    "(close to 0 if the CSV's predictions came from this configuration)",
)
print("\n" + s["caution"] + "\n")
display(expert_rows.drop(columns=["key", "expert_raw"]))

In [ ]:
from coralid.genus_audit import load_legacy_audit
from coralid.portrait_impact import audit_portrait_impact, write_portrait_report

audit = load_legacy_audit(AUDIT_CSV)
audit_rows, audit_summary = audit_portrait_impact(audit, predictions, manifest, GENUS)

a = audit_summary
print(f"{a['n_audit_rows']} audit rows, by orientation {a['rows_by_orientation']}")
print("all judgments:     ", a["all_judgments"])
print("landscape (usable):", a["landscape_judgments"])
print("portrait (mis-cut):", a["portrait_judgments"])
print(
    f"{a['portrait_rows_needing_rereview']} portrait rows need a new review "
    f"({a['portrait_in_new_baseline']} are in the new baseline); "
    f"{a['reusable_judgments']} landscape judgments (yes/no) can be reused"
)
share, ci = a["landscape_share_yes_of_decided"], a["landscape_share_yes_ci95"]
print(f"yes among the decided landscape rows: {share:.3f} (Wilson 95% {ci[0]:.3f}-{ci[1]:.3f})")
display(
    audit_rows[audit_rows["needs_rereview"]].drop(columns=["key", "bbox", "notes"], errors="ignore")
)

report_folder = write_portrait_report(
    expert_rows, expert_summary, cfg.reports_dir, audit_rows, audit_summary
)
print("Saved (a new folder, nothing was overwritten):", report_folder)

## Millepora: the old audit against the new baseline

The old analyst's yes / no / ? judgments joined to the new 20-genus predictions by `(image_name, coral_id)`. The result is reported as **non-expert agreement**, not precision: a judgment agrees with the baseline if the analyst said yes and the baseline calls Millepora, or no and it does not. It is a range, with the unsure counted as wrong and as right. Portrait rows are listed apart, because their old crops were mis-cut.

In [ ]:
from coralid.genus_audit import audit_agreement
from coralid.labeling import summarize_audit

old = summarize_audit(audit, GENUS)
print("The old audit as first reported:", old.counts)
print(
    f"  yes among the decided {old.precision_decided:.3f}; "
    f"{old.precision_unsure_wrong:.3f} to {old.precision_unsure_right:.3f} counting the unsure"
)
print("  what the 'no' crops were:", old.actual_genera)

agreement = audit_agreement(audit, predictions, manifest, GENUS)
print(f"\n{agreement['label']} with the new baseline:")
print(json.dumps(agreement["landscape"], indent=1))
print("\nPortrait rows (NOT comparable, listed for completeness):")
print(json.dumps(agreement["portrait"], indent=1))

## The blind, mixed review sheet

A random sample, across all confidence levels, of the crops the baseline calls Millepora, plus about as many it did not, shuffled, **with no prediction, confidence or runner-up shown**. Crops already judged yes or no on a landscape photo are left out (their judgment is reused). The old audit's unsure crops and every portrait-photo row are put in again for a fresh review. About 40 crops, including every old unsure one, go on a second sheet for Assaf, to measure how often the analyst agrees with him.

Hand out the files in `<run>/reviews/exports/` (a CSV and a PDF per person). **Keep `<run>/reviews/keys/` to yourself.** Reviewers fill `is_millepora` (yes / no / unsure) and, for every `no`, `actual_genus`; completed CSVs go into `<run>/reviews/completed/` under the same file names. Run this cell only once per seed: a second run with the same seed on the same day refuses, as it would overwrite.

In [ ]:
from coralid.genus_audit import build_blind_review
from coralid.masks import AnnotationStore
from coralid.review_export import write_blind_review

N_PRED, N_OTHER, N_ASSAF, SEED = 150, 150, 40, 0  # crops called Millepora, others, for Assaf

review = build_blind_review(
    predictions, manifest, GENUS, audit, n_pred=N_PRED, n_other=N_OTHER, n_assaf=N_ASSAF, seed=SEED
)
print(json.dumps(review.meta, indent=1, default=str))
files = write_blind_review(
    review,
    cfg.reviews_dir / "exports",
    cfg.reviews_dir / "keys",
    AnnotationStore(cfg.json_dir),
    cfg.images_dir,
    crops=cfg.crops,
)
blind_name = files["key"].name.removesuffix("_answer_key.csv")
print("\nname of this review (needed for the scoring):", blind_name)
for what, file in files.items():
    print(f"  {what:12} {file}")

## Scoring the completed reviews

After the analyst (and Assaf, for his subset) have returned their sheets, put them in `<run>/reviews/completed/` named `<name>_analyst.csv` and `<name>_assaf.csv`. This cell ingests them by key, lists anything to fix (a `no` without an `actual_genus`, an unreadable answer, a genus name it does not know), and scores them against the hidden key: precision of the baseline's Millepora calls, the miss rate among its other calls and the recall, with intervals and with the unsure counted both ways; the precision by confidence band; how often the analyst agrees with Assaf; and what the false positives really are. Until the sheets are there it only says what it is waiting for.

In [ ]:
from coralid.labeling import ingest_review
from coralid.review_export import read_blind_key
from coralid.review_scoring import score_blind_review, write_blind_score

SCORE_NAME = None  # later: the name printed above, e.g. "millepora_blind_2026-10-12_seed0"
name = SCORE_NAME or blind_name
analyst_file = cfg.reviews_dir / "completed" / f"{name}_analyst.csv"
assaf_file = cfg.reviews_dir / "completed" / f"{name}_assaf.csv"

if not analyst_file.exists():
    print(f"Waiting for the analyst's completed sheet: {analyst_file}")
else:
    key, meta = read_blind_key(cfg.reviews_dir / "keys", name)
    analyst = ingest_review(analyst_file, meta["answer_column"], cfg.zero_shot.label_set)
    expert_sheet = ingest_review(assaf_file, meta["answer_column"]) if assaf_file.exists() else None
    print("analyst answers:", analyst.counts())
    if expert_sheet is not None:
        print("Assaf answers:  ", expert_sheet.counts())
    else:
        print(f"(no sheet from Assaf yet: {assaf_file})")
    if len(analyst.problems):
        print("\nTo fix in the analyst's sheet:")
        display(analyst.problems)
    score = score_blind_review(analyst, key, meta, expert_sheet)
    print(json.dumps(score.to_dict(), indent=1, default=str))
    print("\nPrecision of the baseline's Millepora calls by confidence band (the random sample):")
    display(score.precision_by_confidence.round(3))
    print("\nSaved (a new folder):", write_blind_score(score, cfg.reports_dir, name))